![image.png](https://i.imgur.com/a3uAqnb.png)
# Lab 5: Video Generation

This notebook covers **video generation** paradigms — temporal consistency, frame interpolation, and connections to world models from the lecture.

You will implement baselines for motion and metrics, then reflect on evaluation and compute cost of modern text-to-video systems.

> 💡 Video adds a **time axis**: coherence across frames is as important as per-frame quality.

__Install dependencies, then work through theory and programming sections.__



# 📦 Installing Required Python Libraries

This cell installs packages needed for this lab.

- **PyTorch / Torchvision** — Tensor video batches and image I/O.
- **Diffusers / Transformers / Accelerate** — Stable Video Diffusion and V-JEPA 2.
- **Decord / imageio** — Video loading and inline playback in Colab.
- **OpenCV / Matplotlib / NumPy** — Metrics and visualization.


In [ ]:
!pip install -q torch torchvision opencv-python matplotlib numpy scipy imageio imageio-ffmpeg decord transformers diffusers accelerate huggingface_hub av

# 📥 Importing Essential Python Libraries

Imports for frame manipulation and temporal metrics in Part B.


In [ ]:
import math
import os
import subprocess
import json

import numpy as np
import matplotlib.pyplot as plt
import torch
from IPython.display import Video, display, HTML

print(f"PyTorch {torch.__version__}")
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = torch.float16 if DEVICE == "cuda" else torch.float32
print(f"Using device: {DEVICE}")

__Theory first — connect definitions to the lecture slides, then move to code.__

---

## 🧠 Part A — Theory

### 📖 A1. Paradigms

1. Contrast **text-to-video diffusion** (Sora-style) models with **latent world prediction** (V-JEPA style) models.
2. Why does video generation have much higher **compute and memory** cost than image generation?

*Write below:*


#### ✍️ Your answers (A1):

1. **Text-to-video diffusion** generates **passive clips** from prompts (cinematic generation). **Genie-style interactive video** is **action-conditioned**, producing **controllable** next frames for agents/playable environments.

2. Video adds a **time dimension** — cost scales with **frames × resolution × denoising steps**, and attention/memory often grows with **spatial-temporal tokens**.


---

## 💻 Part B — Programming
__Let's implement the core ideas in PyTorch.__



### 🛠️ B1. Text-to-video or image-to-video demo

In [ ]:
from diffusers import StableVideoDiffusionPipeline
from diffusers.utils import load_image, export_to_video

pipe = StableVideoDiffusionPipeline.from_pretrained(
    "stabilityai/stable-video-diffusion-img2vid-xt",
    torch_dtype=DTYPE,
    variant="fp16" if DEVICE == "cuda" else None,
)
if DEVICE == "cuda":
    pipe.enable_model_cpu_offload()  # reduce peak GPU memory on Colab T4
else:
    pipe = pipe.to(DEVICE)

init_image = load_image(
    "https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/diffusers/svd/rocket.png"
)
init_image = init_image.resize((384, 384))  # smaller resolution to save memory

frames = pipe(
    init_image,
    num_frames=8,
    decode_chunk_size=2,
    motion_bucket_id=127,
    noise_aug_strength=0.02,
).frames[0]

svd_output_path = "svd_output.mp4"
export_to_video(frames, svd_output_path, fps=4)
print(f"Generated {len(frames)} frames -> {svd_output_path}")
display(HTML("<b>Input image (first frame condition):</b>"))
display(init_image.resize((384, 216)))
display(HTML("<b>Generated video (Stable Video Diffusion):</b>"))
display(Video(svd_output_path, embed=True, width=640))

### 🛠️ B2. V-JEPA 2 Demo

We use a **memory-friendly ViT-L / 256** checkpoint with HuggingFace (fits Colab T4). The official V-JEPA 2 repo is cloned automatically so local PyTorch utilities are available if you extend the lab.

Clone the V-JEPA 2 repository, install imports, and define helper functions. Everything below runs automatically on **Run all**.

In [ ]:
# Clone V-JEPA 2 repo (fixes `ModuleNotFoundError: No module named 'src'`)
if not os.path.exists("vjepa2"):
    subprocess.run(
        ["git", "clone", "--depth", "1", "https://github.com/facebookresearch/vjepa2.git"],
        check=True,
    )
import sys
sys.path.insert(0, "vjepa2")

import torch.nn.functional as F
from decord import VideoReader
from transformers import AutoVideoProcessor, AutoModelForVideoClassification

# Smaller model + fewer frames to avoid OOM on Colab T4
HF_MODEL = "facebook/vjepa2-vitl-fpc16-256-ssv2"
NUM_FRAMES = 16
SAMPLE_VIDEO_PATH = "sample_video.mp4"
SSV2_CLASSES_PATH = "ssv2_classes.json"
SAMPLE_VIDEO_URL = (
    "https://huggingface.co/datasets/nateraw/kinetics-mini/resolve/main/val/bowling/-WH-lxmGJVY_000005_000015.mp4"
)
SSV2_CLASSES_URL = (
    "https://huggingface.co/datasets/huggingface/label-files/resolve/main/something-something-v2-id2label.json"
)


def download_file(url: str, path: str) -> None:
    if not os.path.exists(path):
        subprocess.run(["wget", "-q", url, "-O", path], check=True)
        print(f"Downloaded {path}")


def load_video_frames(path: str, num_frames: int = NUM_FRAMES) -> np.ndarray:
    vr = VideoReader(path)
    frame_idx = np.linspace(0, len(vr) - 1, num_frames, dtype=int)
    return vr.get_batch(frame_idx).asnumpy()  # T x H x W x C


def show_video(path: str, title: str = "Video") -> None:
    display(HTML(f"<b>{title}:</b>"))
    display(Video(path, embed=True, width=640))

Download the sample clip and label map, then **display the input video inline** in the notebook.

In [ ]:
download_file(SAMPLE_VIDEO_URL, SAMPLE_VIDEO_PATH)
download_file(SSV2_CLASSES_URL, SSV2_CLASSES_PATH)
show_video(SAMPLE_VIDEO_PATH, title="Input sample video (bowling clip)")

Load the **ViT-L / 256** V-JEPA 2 classifier from HuggingFace. Weights download automatically — no manual path editing required.

In [ ]:
processor = AutoVideoProcessor.from_pretrained(HF_MODEL)
model = AutoModelForVideoClassification.from_pretrained(HF_MODEL, torch_dtype=DTYPE)
model = model.to(DEVICE).eval()
print(f"Loaded {HF_MODEL} on {DEVICE}")

Run action recognition on the sample video and print the top-5 Something-Something V2 labels.

In [ ]:
video_np = load_video_frames(SAMPLE_VIDEO_PATH, num_frames=NUM_FRAMES)
inputs = processor(list(video_np), return_tensors="pt")
inputs = {k: v.to(DEVICE) for k, v in inputs.items()}

with torch.inference_mode():
    logits = model(**inputs).logits

id2label = json.load(open(SSV2_CLASSES_PATH, "r"))
probs = F.softmax(logits[0], dim=-1)
top5 = probs.topk(5)

print("Top 5 predicted action classes:")
for score, idx in zip(top5.values, top5.indices):
    label = id2label[str(idx.item())]
    print(f"  {label}: {100.0 * score.item():.1f}%")

The clip shows a person putting a bowling ball into a tube, so labels like **"Putting [something] into [something]"** should rank highly.